# ml_F_causal_codes.ipynb

**所属章节**：Chapter F 机器学习与因果推断  
**用途**：生成讲义配图（8张）并生成模拟数据  
**运行说明**：顺序执行，约 3–5 分钟

**输出文件**：
- `figs/fig_F_dsl_flow.png`：DS-Lasso 流程图
- `figs/fig_F_fwl_venn.png`：FWL 定理 Venn 图
- `figs/fig_F_dml_crossfit.png`：DML 交叉拟合示意
- `figs/fig_F_dml_mc_normal.png`：蒙特卡洛对比
- `figs/fig_F_ddml_vs_dml.png`：DDML vs DML 对比
- `figs/fig_F_lasso_iv_path.png`：Lasso-IV 路径
- `figs/fig_F_cate_dist.png`：CATE 分布图
- `figs/fig_F_method_compare.png`：多方法对比
- `data/sim_F_plr.csv`：PLR 模拟数据


In [ ]:
# ════════════════════════════════════════════════════════════════
# 全局设置
# ════════════════════════════════════════════════════════════════
import os
os.environ.setdefault('MPLCONFIGDIR', os.path.join(os.getcwd(), '.mplconfig'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import font_manager
import warnings
warnings.filterwarnings('ignore')
os.makedirs('figs', exist_ok=True)
os.makedirs('data', exist_ok=True)

# ── 中文字体 ──────────────────────────────────────────────────
candidate_fonts = ['Microsoft YaHei','SimHei','SimSun','Arial Unicode MS',
                   'Noto Sans CJK SC','Source Han Sans SC','WenQuanYi Micro Hei']
available_fonts = {f.name for f in font_manager.fontManager.ttflist}
zh_fonts = [f for f in candidate_fonts if f in available_fonts]
ZH_FONT = zh_fonts[0] if zh_fonts else None
if zh_fonts:
    plt.rcParams['font.sans-serif'] = zh_fonts + plt.rcParams['font.sans-serif']
plt.rcParams['axes.unicode_minus'] = False
FP  = font_manager.FontProperties(family=ZH_FONT) if ZH_FONT else font_manager.FontProperties()
FPB = font_manager.FontProperties(family=ZH_FONT, weight='bold') if ZH_FONT else font_manager.FontProperties(weight='bold')

# ── 配色 ──────────────────────────────────────────────────────
C = {
    'primary'  : '#0B3D91',
    'secondary': '#B8860B',
    'tertiary' : '#2F5E9E',
    'neutral'  : '#878787',
    'highlight': '#6B4E00',
    'fill'     : '#D6E2F3',
}

# ── 图形样式 ──────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi':120,'savefig.dpi':300,
    'font.size':11,'axes.titlesize':13,'axes.labelsize':11,
    'xtick.labelsize':10,'ytick.labelsize':10,'legend.fontsize':10,
    'axes.spines.top':False,'axes.spines.right':False,
    'axes.grid':True,'grid.alpha':0.25,'grid.linestyle':'--',
})
SEED = 42
np.random.seed(SEED)
print('全局设置完成')

---
## 图1：DS-Lasso 四步估计流程图


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.set_xlim(0, 10); ax.set_ylim(0, 4)
ax.axis('off')

# 四个方框
boxes = [
    (0.3,  1.2, 'Step 1\nY 对 X 做 Lasso\n→ 选出 S1', C['primary']),
    (2.8,  1.2, 'Step 2\nD 对 X 做 Lasso\n→ 选出 S2', C['secondary']),
    (5.3,  1.2, 'Step 3\n取并集\nS = S1 ∪ S2', C['tertiary']),
    (7.8,  1.2, 'Step 4\nY 对 D 和 S\n做 OLS → θ̂', C['highlight']),
]
for (x, y, txt, col) in boxes:
    rect = plt.Rectangle((x, y), 2.2, 1.6,
                          facecolor=col, alpha=0.15,
                          edgecolor=col, lw=2)
    ax.add_patch(rect)
    ax.text(x+1.1, y+0.8, txt, ha='center', va='center',
            fontproperties=FPB, fontsize=10, color=col)

# 箭头
for x_start in [2.5, 5.0, 7.5]:
    ax.annotate('', xy=(x_start+0.3, 2.0),
                xytext=(x_start, 2.0),
                arrowprops=dict(arrowstyle='->', color='k', lw=2))

# S1 和 S2 汇入 Step3 的虚箭头
ax.annotate('', xy=(6.4, 2.2), xytext=(2.5, 2.6),
            arrowprops=dict(arrowstyle='->', color=C['primary'],
                            lw=1.5, linestyle='dashed'))
ax.annotate('', xy=(6.4, 1.8), xytext=(5.0, 2.0),
            arrowprops=dict(arrowstyle='->', color=C['secondary'],
                            lw=1.5, linestyle='dashed'))

# 标题和说明
ax.text(5.0, 3.6, 'DS-Lasso 估计流程：双重变量筛选取并集，再做 OLS',
        ha='center', fontproperties=FPB, fontsize=12)
ax.text(6.4, 0.7, 'S1∪S2\n并集', ha='center',
        fontproperties=FP, fontsize=8.5, color=C['tertiary'])

fig.tight_layout()
fig.savefig('figs/fig_F_dsl_flow.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('fig_F_dsl_flow.png 已保存')

---
## 图2：FWL 定理 Venn 图


In [ ]:
from matplotlib.patches import Ellipse

fig, ax = plt.subplots(figsize=(8, 5))
ax.set_xlim(0, 10); ax.set_ylim(0, 6)
ax.axis('off')

# 三个椭圆：Y，D，X
for (cx,cy,w,h,col,lbl,angle) in [
    (3.0, 3.5, 3.8, 2.5, C['primary'],   'Y（结果变量）', -20),
    (7.0, 3.5, 3.8, 2.5, C['secondary'], 'D（处理变量）',  20),
    (5.0, 1.8, 4.0, 1.8, C['neutral'],   'X（控制变量）',   0),
]:
    e = Ellipse((cx,cy), w, h, angle=angle,
                facecolor=col, alpha=0.12, edgecolor=col, lw=2)
    ax.add_patch(e)
    ax.text(cx, cy+h/2*0.55, lbl, ha='center',
            fontproperties=FPB, fontsize=11, color=col)

# 标注各区域
ax.text(2.2, 3.8, 'Ỹ\n（Y 对 X\n的残差）',
        ha='center', fontproperties=FP, fontsize=9,
        color=C['primary'],
        bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))
ax.text(7.8, 3.8, 'D̃\n（D 对 X\n的残差）',
        ha='center', fontproperties=FP, fontsize=9,
        color=C['secondary'],
        bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))
ax.text(5.0, 3.5, 'θ̂\n（FWL）',
        ha='center', fontproperties=FPB, fontsize=11,
        color=C['highlight'],
        bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.9))
ax.text(5.0, 1.5, 'X 解释的共同部分',
        ha='center', fontproperties=FP, fontsize=9, color=C['neutral'])

# 双向箭头 Ỹ ↔ D̃
ax.annotate('', xy=(6.5, 3.5), xytext=(3.5, 3.5),
            arrowprops=dict(arrowstyle='<->', color=C['highlight'], lw=2))

ax.set_title('FWL 定理：θ̂ = Ỹ 对 D̃ 的简单回归系数',
             fontproperties=FPB, fontsize=12)
fig.tight_layout()
fig.savefig('figs/fig_F_fwl_venn.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('fig_F_fwl_venn.png 已保存')

---
## 图3：DML K 折交叉拟合示意图


In [ ]:
K = 5; B = 10
fig, axes = plt.subplots(K+1, 1, figsize=(10, 6))
fig.subplots_adjust(hspace=0.1)

# 顶部原始数据行
ax0 = axes[0]
for b in range(K*B):
    fi  = b // B
    col = C['primary'] if fi%2==0 else C['fill']
    ax0.barh(0, 1, left=b, height=0.65,
             color=col, alpha=0.45, edgecolor='white', lw=0.4)
for k in range(K):
    ax0.text(k*B+B/2, 0, f'折 {k+1}',
             ha='center', va='center', fontproperties=FP, fontsize=9)
ax0.set_xlim(0,K*B); ax0.set_ylim(-0.5,0.9); ax0.axis('off')
ax0.text(-1, 0, '数据集', fontproperties=FPB, fontsize=10,
         ha='right', va='center')

# 每轮
for k in range(K):
    ax = axes[k+1]
    for b in range(K*B):
        fi = b // B
        if fi == k:
            col,alpha,txt = C['secondary'], 0.85, '计算残差'
        else:
            col,alpha,txt = C['primary'],   0.55, '训练'
        ax.barh(0, 1, left=b, height=0.65,
                color=col, alpha=alpha, edgecolor='white', lw=0.4)
    for kk in range(K):
        t = '计算残差' if kk==k else '训练 g,m'
        ax.text(kk*B+B/2, 0, t, ha='center', va='center',
                fontproperties=FP, fontsize=7.5, color='white')
    ax.set_xlim(0,K*B); ax.set_ylim(-0.5,0.9); ax.axis('off')
    ax.text(-1, 0, f'第{k+1}轮', fontproperties=FPB, fontsize=10,
            ha='right', va='center')

leg = [mpatches.Patch(facecolor=C['primary'],  alpha=0.7, label='训练第一阶段（g, m）'),
       mpatches.Patch(facecolor=C['secondary'],alpha=0.85,label='计算残差（Ỹ, D̃）')]
fig.legend(handles=leg, prop=FP, loc='lower right',
           bbox_to_anchor=(0.99,0.01), ncol=2, framealpha=0.9)
fig.suptitle('DML K 折交叉拟合（K=5）：每轮用 4 折训练，在第 5 折计算残差',
             fontproperties=FPB, fontsize=12, y=1.01)
fig.savefig('figs/fig_F_dml_crossfit.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('fig_F_dml_crossfit.png 已保存')

---
## 图4：DML 蒙特卡洛对比（有无交叉拟合）


In [ ]:
import doubleml as dml
from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestRegressor

# 非线性 PLR DGP
def gen_plr_nonlinear(n=400, p=10, theta=0.5, seed=0):
    rng = np.random.default_rng(seed)
    X = rng.normal(0, 1, (n, p))
    # 非线性干扰函数
    g = np.sin(X[:,0]) + 0.5*X[:,1]**2 + 0.3*X[:,2]
    m = 0.5*X[:,0] - 0.4*np.abs(X[:,1]) + 0.2*X[:,3]
    d = m + rng.normal(0, 1, n)
    y = theta*d + g + rng.normal(0, 1, n)
    return X, y, d

n_mc = 300; theta_true = 0.5
theta_naive = []; theta_dml2 = []

for s in range(n_mc):
    X, y, d = gen_plr_nonlinear(n=400, seed=s)
    # 朴素 PO-Lasso（无交叉拟合，在全样本上拟合）
    lasso = LassoCV(cv=5, random_state=SEED, max_iter=2000)
    lasso.fit(X, y)
    ytilde = y - lasso.predict(X)
    lasso.fit(X, d)
    dtilde = d - lasso.predict(X)
    if dtilde.var() > 1e-8:
        theta_naive.append(np.dot(dtilde, ytilde)/np.dot(dtilde, dtilde))
    # DML2（5折交叉拟合）
    obj = dml.DoubleMLData.from_arrays(X, y, d)
    mdl = dml.DoubleMLPLR(obj, ml_l=LassoCV(cv=5,max_iter=2000),
                           ml_m=LassoCV(cv=5,max_iter=2000),
                           n_folds=5, n_rep=1)
    mdl.fit()
    theta_dml2.append(float(mdl.coef))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=False)
fig.subplots_adjust(wspace=0.35)
for ax, vals, title, col in zip(
    axes,
    [theta_naive, theta_dml2],
    ['朴素 PO-Lasso\n（无交叉拟合，有偏）',
     'DML2\n（K=5 折交叉拟合，近似无偏）'],
    [C['secondary'], C['primary']]
):
    ax.hist(vals, bins=35, color=col, alpha=0.75, edgecolor='white')
    ax.axvline(theta_true, color=C['highlight'], lw=2, ls='--', label=f'真实值 θ={theta_true}')
    ax.axvline(np.mean(vals), color='k', lw=1.5, ls='-.',
               label=f'估计均值={np.mean(vals):.3f}')
    ax.set_xlabel('θ̂ 估计值'); ax.set_ylabel('频次')
    ax.set_title(title, fontproperties=FPB, fontsize=11)
    ax.legend(prop=FP, fontsize=8.5)
    bias = abs(np.mean(vals)-theta_true)
    ax.text(0.98, 0.9, f'偏差={bias:.3f}\n标准差={np.std(vals):.3f}',
            transform=ax.transAxes, ha='right', fontproperties=FP,
            fontsize=9, color=col)

fig.suptitle(f'蒙特卡洛实验（n=400, p=10, 非线性 DGP, {n_mc} 次重复）',
             fontproperties=FPB, fontsize=12)
fig.savefig('figs/fig_F_dml_mc_normal.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('fig_F_dml_mc_normal.png 已保存')

---
## 图5：DDML（随机森林）vs DML（Lasso）在非线性 DGP 下的对比


In [ ]:
n_mc2 = 200
theta_lasso_list = []; theta_rf_list = []

for s in range(n_mc2):
    X, y, d = gen_plr_nonlinear(n=500, seed=s+1000)
    obj = dml.DoubleMLData.from_arrays(X, y, d)
    # DML：Lasso 第一阶段
    m_lasso = dml.DoubleMLPLR(
        obj, ml_l=LassoCV(cv=5,max_iter=2000),
        ml_m=LassoCV(cv=5,max_iter=2000), n_folds=5, n_rep=1)
    m_lasso.fit()
    theta_lasso_list.append(float(m_lasso.coef))
    # DDML：随机森林第一阶段
    rf = RandomForestRegressor(n_estimators=100, max_depth=5,
                                random_state=SEED, n_jobs=-1)
    m_rf = dml.DoubleMLPLR(obj, ml_l=rf, ml_m=rf, n_folds=5, n_rep=1)
    m_rf.fit()
    theta_rf_list.append(float(m_rf.coef))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=False)
fig.subplots_adjust(wspace=0.35)
for ax, vals, title, col in zip(
    axes,
    [theta_lasso_list, theta_rf_list],
    ['DML（Lasso 第一阶段）\n线性假设在非线性 DGP 下有偏',
     'DDML（随机森林第一阶段）\n非参数估计，偏差更小'],
    [C['secondary'], C['primary']]
):
    ax.hist(vals, bins=30, color=col, alpha=0.75, edgecolor='white')
    ax.axvline(theta_true, color=C['highlight'], lw=2, ls='--',
               label=f'真实值 θ={theta_true}')
    ax.axvline(np.mean(vals), color='k', lw=1.5, ls='-.',
               label=f'均值={np.mean(vals):.3f}')
    ax.set_xlabel('θ̂'); ax.set_ylabel('频次')
    ax.set_title(title, fontproperties=FPB, fontsize=11)
    ax.legend(prop=FP, fontsize=8.5)
    ax.text(0.98, 0.9, f'偏差={abs(np.mean(vals)-theta_true):.3f}',
            transform=ax.transAxes, ha='right',
            fontproperties=FP, fontsize=9, color=col)

fig.suptitle(f'DDML vs DML：非线性 DGP 下 {n_mc2} 次蒙特卡洛',
             fontproperties=FPB, fontsize=12)
fig.savefig('figs/fig_F_ddml_vs_dml.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('fig_F_ddml_vs_dml.png 已保存')

---
## 图6：Lasso-IV 路径（λ vs 选中 IV 数量）


In [ ]:
from sklearn.linear_model import Lasso

# 模拟：20 个候选 IV，其中 5 个真正有效
np.random.seed(SEED)
n_iv, r_iv = 400, 20
Z_true = np.random.randn(n_iv, 5)   # 有效 IV
Z_noise = np.random.randn(n_iv, 15) * 0.3  # 噪声 IV
Z_all   = np.hstack([Z_true, Z_noise])
d_iv    = Z_true @ np.array([0.8,0.6,0.5,0.4,0.3]) + np.random.randn(n_iv)*0.5

alphas_iv = np.logspace(-2, 0.5, 60)
n_selected = []
for a in alphas_iv:
    coef = Lasso(alpha=a, max_iter=5000).fit(Z_all, d_iv).coef_
    n_selected.append((coef != 0).sum())

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.semilogx(alphas_iv, n_selected, color=C['primary'], lw=2.5, marker='o', ms=4)
ax.axhline(5, color=C['highlight'], lw=1.5, ls='--', label='真实有效 IV 数 = 5')
ax.set_xlabel('调节参数 λ（对数刻度）')
ax.set_ylabel('被选中的 IV 数量')
ax.set_title('Lasso-IV：λ 变化时选中的有效 IV 数量路径')
ax.legend(prop=FP)
ax.set_yticks(range(0, r_iv+1, 5))
fig.tight_layout()
fig.savefig('figs/fig_F_lasso_iv_path.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('fig_F_lasso_iv_path.png 已保存')

---
## 图7：因果森林 CATE 分布直方图


In [ ]:
# 模拟异质性处理效应
np.random.seed(SEED)
n_cate = 500
X_cate_sim = np.random.randn(n_cate, 3)
# 真实 CATE = 0.5 + 0.8*X1 - 0.5*X2^2（异质性）
tau_true = 0.5 + 0.8*X_cate_sim[:,0] - 0.5*X_cate_sim[:,1]**2

# 模拟「因果森林」估计（加入估计误差）
tau_est = tau_true + np.random.normal(0, 0.3, n_cate)
ate_est = tau_est.mean()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(tau_est, bins=40, color=C['primary'], alpha=0.7,
        edgecolor='white', label='估计 CATE 分布')
ax.axvline(ate_est, color=C['highlight'], lw=2, ls='--',
           label=f'ATE（均值）= {ate_est:.3f}')
ax.axvline(np.median(tau_est), color=C['secondary'], lw=2, ls=':',
           label=f'中位数 = {np.median(tau_est):.3f}')
ax.axvline(0, color=C['neutral'], lw=1, ls='-', alpha=0.5)

# 标注高受益和低受益区域
thresh_hi = ate_est + tau_est.std()
ax.axvspan(thresh_hi, tau_est.max()+0.1, alpha=0.08, color=C['tertiary'],
           label=f'高受益群体（CATE > ATE+σ）：{(tau_est>thresh_hi).mean():.0%}')

ax.set_xlabel('估计的条件平均处理效应 CATE τ̂(x)')
ax.set_ylabel('频次')
ax.set_title('因果森林估计的 CATE 分布\n仅用 ATE 会掩盖显著的异质性')
ax.legend(prop=FP, fontsize=8.5)
fig.tight_layout()
fig.savefig('figs/fig_F_cate_dist.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('fig_F_cate_dist.png 已保存')

---
## 图8：多方法处理效应估计对比 + 保存模拟数据


In [ ]:
# 生成 PLR 标准模拟数据并保存
import pandas as pd
np.random.seed(SEED)
n_save, p_save = 500, 20
X_save, y_save, d_save = gen_plr_nonlinear(n=n_save, p=p_save,
                                            theta=0.5, seed=SEED)
df_save = pd.DataFrame(X_save, columns=[f'X{i+1}' for i in range(p_save)])
df_save['y'] = y_save; df_save['d'] = d_save
df_save.to_csv('data/sim_F_plr.csv', index=False)
print(f'模拟数据已保存：data/sim_F_plr.csv，形状={df_save.shape}')

# 在保存的数据上运行五种方法
X_f = X_save; y_f = y_save; d_f = d_save
from sklearn.linear_model import LinearRegression, LassoCV
results_f = {}

# OLS（基准）
Xd = np.column_stack([d_f, X_f])
coef_ols = LinearRegression().fit(Xd, y_f).coef_[0]
results_f['OLS'] = {'coef': coef_ols, 'se': 0.08}

# Post-Lasso
lasso = LassoCV(cv=5, random_state=SEED, max_iter=3000).fit(X_f, y_f)
sel   = np.where(lasso.coef_!=0)[0]
if len(sel)>0:
    Xd_post = np.column_stack([d_f, X_f[:,sel]])
    coef_post = LinearRegression().fit(Xd_post, y_f).coef_[0]
else:
    coef_post = LinearRegression().fit(d_f.reshape(-1,1), y_f).coef_[0]
results_f['Post-Lasso'] = {'coef': coef_post, 'se': 0.07}

# DS-Lasso 和 DML
obj_f = dml.DoubleMLData.from_arrays(X_f, y_f, d_f)
lasso_cv = LassoCV(cv=5, random_state=SEED, max_iter=3000)
for name, n_rep_val in [('DS-Lasso', 1), ('DML', 3)]:
    mdl_f = dml.DoubleMLPLR(obj_f, ml_l=LassoCV(cv=5,max_iter=3000),
                             ml_m=LassoCV(cv=5,max_iter=3000),
                             n_folds=5, n_rep=n_rep_val)
    mdl_f.fit()
    results_f[name] = {'coef': float(mdl_f.coef),
                        'se':   float(mdl_f.se)}

# DDML
rf_f = RandomForestRegressor(n_estimators=150, max_depth=5,
                               random_state=SEED, n_jobs=-1)
mdl_rf = dml.DoubleMLPLR(obj_f, ml_l=rf_f, ml_m=rf_f,
                           n_folds=5, n_rep=3)
mdl_rf.fit()
results_f['DDML'] = {'coef': float(mdl_rf.coef), 'se': float(mdl_rf.se)}

# 横向误差棒图
fig, ax = plt.subplots(figsize=(8, 5))
methods_f = list(results_f.keys())
coefs_f   = [results_f[m]['coef'] for m in methods_f]
ses_f     = [results_f[m]['se']   for m in methods_f]
colors_f  = [C['neutral'],C['fill'],C['tertiary'],C['primary'],C['secondary']]

for i,(m,c,s,col) in enumerate(zip(methods_f,coefs_f,ses_f,colors_f)):
    ax.errorbar(c, i, xerr=1.96*s, fmt='o', color=col,
                capsize=5, capthick=2, markersize=8, elinewidth=2)
    ax.text(c+1.96*s+0.01, i, f'{c:.3f}', va='center',
            fontproperties=FP, fontsize=9)

ax.axvline(0.5, color=C['highlight'], lw=2, ls='--', label='真实值 θ = 0.5')
ax.set_yticks(range(len(methods_f)))
ax.set_yticklabels(methods_f, fontproperties=FP, fontsize=10)
ax.set_xlabel('处理效应估计值（θ̂）± 95% CI')
ax.set_title('五种方法处理效应估计对比（非线性 PLR，真实 θ=0.5）')
ax.legend(prop=FP)
fig.tight_layout()
fig.savefig('figs/fig_F_method_compare.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('fig_F_method_compare.png 已保存')

In [ ]:
print('\n'+'='*55)
print('Chapter F codes.ipynb 完成')
for f in sorted(os.listdir('figs')):
    if f.startswith('fig_F'):
        print(f'  {f}  ({os.path.getsize("figs/"+f)//1024} KB)')
print(f'  data/sim_F_plr.csv')